In [ ]:
import pyarrow.parquet as pq

pf = pq.ParquetFile("/app/data/raw/COMPACTED_nyct_mta_bus_gtfsrt_2026-10-16_18_03_57.parquet")
print(pf.schema_arrow)          # columns and types (watch for nested structs)
print(pf.metadata)              # rows, row groups, size

id: string
vehicle.trip.trip_id: string
vehicle.trip.start_date: string
vehicle.trip.route_id: string
vehicle.trip.direction_id: double
vehicle.position.latitude: double
vehicle.position.longitude: double
vehicle.position.bearing: double
vehicle.timestamp: timestamp[ns]
vehicle.stop_id: string
vehicle.vehicle.id: string
vehicle.occupancy_status: double
-- schema metadata --
pandas: '{"column_indexes": [{"field_name": null, "metadata": null, "name' + 2001
  created_by: fastparquet-python version 2024.2.0 (build 0)
  num_columns: 12
  num_rows: 3054703
  num_row_groups: 1
  format_version: 1.0
  serialized_size: 3515


In [2]:
import pandas as pd

df = pf.read_row_group(0).to_pandas()   # or df.head(50_000)
display(df.head(10))
print(df.dtypes)
print(df.isna().mean().sort_values(ascending=False))   # null rates
print(df.nunique().sort_values())                       # distinct counts
print(df.duplicated().sum())                            # exact duplicates

,id,vehicle.trip.trip_id,vehicle.trip.start_date,vehicle.trip.route_id,vehicle.trip.direction_id,vehicle.position.latitude,vehicle.position.longitude,vehicle.position.bearing,vehicle.timestamp,vehicle.stop_id,vehicle.vehicle.id,vehicle.occupancy_status
0,MTA NYCT_9771,MQ_D6-Weekday-SDon-078100_M57_460,20261015,M57,0.0,40.756630,-73.962982,157.029770,2026-10-15 18:01:30+00:00,903302,MTA NYCT_9771,1.0
1,MTA NYCT_8440,JA_D6-Weekday-SDon-079800_MISC_742,20261015,Q77,1.0,40.668270,-73.756790,250.436829,2026-10-15 18:01:27+00:00,552362,MTA NYCT_8440,NaN
2,MTA NYCT_7112,EN_D6-Weekday-SDon-076000_B82_629,20261015,B82,1.0,40.605904,-73.986259,172.034500,2026-10-15 18:01:29+00:00,300486,MTA NYCT_7112,1.0
3,MTA NYCT_9775,MV_D6-Weekday-SDon-073500_M5_501,20261015,M4,1.0,40.761791,-73.974815,234.039490,2026-10-15 18:01:47+00:00,403266,MTA NYCT_9775,1.0
4,MTA NYCT_8443,JA_D6-Weekday-SDon-081000_Q5_209,20261015,Q5,1.0,40.702869,-73.797760,194.153412,2026-10-15 18:01:40+00:00,503172,MTA NYCT_8443,NaN
5,MTA NYCT_9774,MQ_D6-Weekday-SDon-084900_M42_307,20261015,M42,0.0,40.762310,-74.001350,214.830353,2026-10-15 18:01:36+00:00,401812,MTA NYCT_9774,1.0
6,MTA NYCT_9773,MQ_D6-Weekday-SDon-082200_M42_312,20261015,M42,0.0,40.752438,-73.978363,336.501434,2026-10-15 18:01:21+00:00,403459,MTA NYCT_9773,1.0
7,MTA NYCT_9772,GH_D6-Weekday-SDon-081400_BX16_309,20261015,BX16,0.0,40.896458,-73.869164,350.439911,2026-10-15 18:01:19+00:00,103662,MTA NYCT_9772,1.0
8,MTA NYCT_7111,EN_D6-Weekday-SDon-084100_B42_508,20261015,B42,1.0,40.644623,-73.901733,318.060822,2026-10-15 18:01:36+00:00,306590,MTA NYCT_7111,NaN
9,MTA NYCT_9779,MV_D6-Weekday-SDon-081600_M3_335,20261015,M3,1.0,40.841721,-73.939377,233.130096,2026-10-15 18:01:36+00:00,400258,MTA NYCT_9779,1.0


id                                            str
vehicle.trip.trip_id                          str
vehicle.trip.start_date                       str
vehicle.trip.route_id                         str
vehicle.trip.direction_id                 float64
vehicle.position.latitude                 float64
vehicle.position.longitude                float64
vehicle.position.bearing                  float64
vehicle.timestamp             datetime64[ns, UTC]
vehicle.stop_id                               str
vehicle.vehicle.id                            str
vehicle.occupancy_status                  float64
dtype: object
vehicle.occupancy_status      0.576394
id                            0.000000
vehicle.trip.start_date       0.000000
vehicle.trip.trip_id          0.000000
vehicle.trip.route_id         0.000000
vehicle.trip.direction_id     0.000000
vehicle.position.longitude    0.000000
vehicle.position.latitude     0.000000
vehicle.position.bearing      0.000000
vehicle.timestamp             0.000

In [14]:
df.columns

Index(['id', 'vehicle.trip.trip_id', 'vehicle.trip.start_date',
       'vehicle.trip.route_id', 'vehicle.trip.direction_id',
       'vehicle.position.latitude', 'vehicle.position.longitude',
       'vehicle.position.bearing', 'vehicle.timestamp', 'vehicle.stop_id',
       'vehicle.vehicle.id', 'vehicle.occupancy_status'],
      dtype='str')

In [10]:
import pyarrow.parquet as pq
import pandas as pd
from pathlib import Path

FILE = Path("/app/data/raw/COMPACTED_nyct_mta_bus_gtfsrt_2026-10-16_18_03_57.parquet")
pf = pq.ParquetFile(FILE)

print("file size (MB):", round(FILE.stat().st_size / 1e6, 2))
print("rows:", pf.metadata.num_rows, "| row groups:", pf.metadata.num_row_groups)

df = pq.read_table(FILE).to_pandas()      # fine if the file is small (under ~200 MB)

ts = pd.to_datetime(df["vehicle.timestamp"], utc=True)
print("UTC:", ts.min(), "->", ts.max())
print("NY :", ts.min().tz_convert("America/New_York"), "->", ts.max().tz_convert("America/New_York"))

file size (MB): 134.06
rows: 3054703 | row groups: 1
UTC: 2026-10-15 18:00:04+00:00 -> 2026-10-16 18:01:00+00:00
NY : 2026-10-15 14:00:04-04:00 -> 2026-10-16 14:01:00-04:00


In [12]:
print("distinct buses :", df["vehicle.vehicle.id"].nunique())
print("distinct trips :", df["vehicle.trip.trip_id"].nunique())
print("distinct routes:", df["vehicle.trip.route_id"].nunique())

print("\nid == vehicle id:", (df["id"] == df["vehicle.vehicle.id"]).mean())
print("exact duplicate rows:", df.duplicated().sum())
print("duplicate (bus, timestamp):", df.duplicated(["vehicle.vehicle.id", "vehicle.timestamp"]).sum())

print("\nlatitude :", df["vehicle.position.latitude"].min(), df["vehicle.position.latitude"].max())
print("longitude:", df["vehicle.position.longitude"].min(), df["vehicle.position.longitude"].max())

print("\noccupancy values:")
print(df["vehicle.occupancy_status"].value_counts(dropna=False))

distinct buses : 5137
distinct trips : 53906
distinct routes: 347

id == vehicle id: 1.0
exact duplicate rows: 1267
duplicate (bus, timestamp): 8014

latitude : 40.50288772583008 40.933719635009766
longitude: -74.25233459472656 -73.70143127441406

occupancy values:
vehicle.occupancy_status
NaN    1760712
1.0    1034215
2.0     225338
3.0      29590
5.0       4848
Name: count, dtype: int64


In [13]:
descriptions = {
    "id": "Row ID in the feed (equal to the bus ID if the check above says 1.0)",
    "vehicle.trip.trip_id": "One scheduled run of a bus; key to the static GTFS schedule",
    "vehicle.trip.start_date": "Service date of the trip (YYYYMMDD)",
    "vehicle.trip.route_id": "Bus line, e.g. M57",
    "vehicle.trip.direction_id": "Direction of travel (0 or 1); float because of missing values",
    "vehicle.position.latitude": "Latitude in degrees",
    "vehicle.position.longitude": "Longitude in degrees",
    "vehicle.position.bearing": "Heading in degrees (0 = north, 90 = east)",
    "vehicle.timestamp": "When the position was measured (UTC)",
    "vehicle.stop_id": "Stop related to the bus (at or heading to; status not given)",
    "vehicle.vehicle.id": "Bus ID",
    "vehicle.occupancy_status": "How full the bus is, as a numeric code",
}

lines = ["# Data Dictionary", "",
         f"Source file: `{FILE.name}`  |  rows: {len(df)}", "",
         "| Column | Type | Meaning | Null % | Example |",
         "|---|---|---|---|---|"]

for col in df.columns:
    non_null = df[col].dropna()
    example = non_null.iloc[0] if len(non_null) else ""
    lines.append(f"| `{col}` | {df[col].dtype} | {descriptions.get(col, '')} | "
                 f"{df[col].isna().mean() * 100:.2f} | {example} |")

print("\n".join(lines))

# Data Dictionary

Source file: `COMPACTED_nyct_mta_bus_gtfsrt_2026-10-16_18_03_57.parquet`  |  rows: 3054703

| Column | Type | Meaning | Null % | Example |
|---|---|---|---|---|
| `id` | str | Row ID in the feed (equal to the bus ID if the check above says 1.0) | 0.00 | MTA NYCT_9771 |
| `vehicle.trip.trip_id` | str | One scheduled run of a bus; key to the static GTFS schedule | 0.00 | MQ_D6-Weekday-SDon-078100_M57_460 |
| `vehicle.trip.start_date` | str | Service date of the trip (YYYYMMDD) | 0.00 | 20261015 |
| `vehicle.trip.route_id` | str | Bus line, e.g. M57 | 0.00 | M57 |
| `vehicle.trip.direction_id` | float64 | Direction of travel (0 or 1); float because of missing values | 0.00 | 0.0 |
| `vehicle.position.latitude` | float64 | Latitude in degrees | 0.00 | 40.756629943847656 |
| `vehicle.position.longitude` | float64 | Longitude in degrees | 0.00 | -73.96298217773438 |
| `vehicle.position.bearing` | float64 | Heading in degrees (0 = north, 90 = east) | 0.00 | 157.029769897460

Step 2: Download Manhattan gtfs_static

In [ ]:
from pathlib import Path
import requests, zipfile
import pandas as pd

GTFS_DIR = Path("/app/data/raw/gtfs_static")
GTFS_DIR.mkdir(parents=True, exist_ok=True)

url = "http://web.mta.info/developers/data/nyct/bus/google_transit_manhattan.zip"
zip_path = GTFS_DIR / "google_transit_manhattan.zip"

# 1. download (stream to disk, don't hold it all in memory)
with requests.get(url, stream=True, timeout=120) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            f.write(chunk)
print("downloaded:", round(zip_path.stat().st_size / 1e6, 1), "MB")

# 2. extract and list the files with their sizes
out = GTFS_DIR / "manhattan"
with zipfile.ZipFile(zip_path) as z:
    z.extractall(out)
for p in sorted(out.glob("*.txt")):
    print(p.name, round(p.stat().st_size / 1e6, 1), "MB")

# 3. schedule validity dates (if the file exists)
fi = out / "feed_info.txt"
if fi.exists():
    print(pd.read_csv(fi, dtype=str).T)

In [ ]:
trips = pd.read_csv(out / "trips.txt", dtype=str)

rt = df["vehicle.trip.trip_id"].dropna().unique()     # df = your realtime dataframe
found = pd.Series(rt).isin(trips["trip_id"])
print(f"{found.mean():.1%} of all realtime trip_ids are in Manhattan trips.txt")

# Manhattan routes start with M, so test those separately
m = df[df["vehicle.trip.route_id"].astype(str).str.startswith("M")]
m_rt = m["vehicle.trip.trip_id"].dropna().unique()
print(f"{pd.Series(m_rt).isin(trips['trip_id']).mean():.1%} of the M-route trip_ids match")

In [17]:
out = Path("/app/data/raw/gtfs_static/manhattan")
trips = pd.read_csv(out / "trips.txt", dtype=str)

# all realtime trip_ids
rt = df["vehicle.trip.trip_id"].dropna().unique()
print(f"All routes: {pd.Series(rt).isin(trips['trip_id']).mean():.1%} of trip_ids are in Manhattan trips.txt")

# Manhattan routes only (they start with M)
m = df[df["vehicle.trip.route_id"].astype(str).str.startswith("M")]
m_rt = m["vehicle.trip.trip_id"].dropna().unique()
print(f"M routes only: {pd.Series(m_rt).isin(trips['trip_id']).mean():.1%} of {len(m_rt)} trip_ids match")

All routes: 15.3% of trip_ids are in Manhattan trips.txt
M routes only: 97.3% of 8492 trip_ids match


Step 2: Get this trip’s scheduled stops

In [20]:
import numpy as np
import pandas as pd

route = "M57"   # any Manhattan route you saw in your data
d = df[df["vehicle.trip.route_id"] == route].copy()
d = d.drop_duplicates(["vehicle.trip.trip_id", "vehicle.trip.start_date", "vehicle.timestamp"])

counts = (d.groupby(["vehicle.trip.trip_id", "vehicle.trip.start_date"])
            .size().sort_values(ascending=False))
print(counts.head(10))      # trips with the most pings

trip_id, start_date = counts.index[3]     # pick one near the top, change the number to try another
pings = (d[(d["vehicle.trip.trip_id"] == trip_id) & (d["vehicle.trip.start_date"] == start_date)]
         .sort_values("vehicle.timestamp").reset_index(drop=True))

print(trip_id, start_date, len(pings), "pings")
print(pings["vehicle.timestamp"].diff().describe())     # time between pings

vehicle.trip.trip_id               vehicle.trip.start_date
MQ_D6-Weekday-SDon-073000_M57_460  20261016                   66
MQ_D6-Weekday-SDon-064000_M72_603  20261016                   63
MQ_D6-Weekday-SDon-138300_M72_617  20261015                   63
MQ_D6-Weekday-SDon-099300_M57_464  20261015                   63
MQ_D6-Weekday-SDon-089300_M72_609  20261015                   62
MQ_D6-Weekday-SDon-067000_M57_459  20261016                   61
MQ_D6-Weekday-SDon-093800_M57_464  20261015                   61
MQ_D6-Weekday-SDon-085900_M57_455  20261015                   59
MQ_D6-Weekday-SDon-071500_M57_458  20261016                   59
MQ_D6-Weekday-SDon-106300_M72_610  20261015                   59
dtype: int64
MQ_D6-Weekday-SDon-099300_M57_464 20261015 63 pings
count                           62
mean     0 days 00:01:02.983870967
std      0 days 00:00:25.422006689
min                0 days 00:00:53
25%                0 days 00:00:54
50%         0 days 00:00:54.500000
75%         0 da

In [21]:
st_chunks = pd.read_csv(out / "stop_times.txt", dtype=str,
                        usecols=["trip_id", "arrival_time", "stop_id", "stop_sequence"],
                        chunksize=500_000)
st = pd.concat(c[c["trip_id"] == trip_id] for c in st_chunks)

stops = pd.read_csv(out / "stops.txt", dtype=str, usecols=["stop_id", "stop_lat", "stop_lon"])
stops[["stop_lat", "stop_lon"]] = stops[["stop_lat", "stop_lon"]].astype(float)

st = st.merge(stops, on="stop_id", how="left")
st["stop_sequence"] = st["stop_sequence"].astype(int)
st = st.sort_values("stop_sequence").reset_index(drop=True)
print(len(st), "scheduled stops")
st.head()

22 scheduled stops


,trip_id,arrival_time,stop_id,stop_sequence,stop_lat,stop_lon
0,MQ_D6-Weekday-SDon-099300_M57_464,16:33:00,400751,1,40.779127,-73.982766
1,MQ_D6-Weekday-SDon-099300_M57_464,16:34:37,403036,2,40.779534,-73.984810
2,MQ_D6-Weekday-SDon-099300_M57_464,16:36:42,403037,3,40.777468,-73.986337
3,MQ_D6-Weekday-SDon-099300_M57_464,16:38:00,403038,4,40.776180,-73.987276
4,MQ_D6-Weekday-SDon-099300_M57_464,16:39:28,404826,5,40.774490,-73.988410


In [22]:
service_midnight = pd.Timestamp(start_date, tz="America/New_York")
st["scheduled"] = (service_midnight + pd.to_timedelta(st["arrival_time"])).dt.tz_convert("UTC")

In [23]:
def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

lat = pings["vehicle.position.latitude"].to_numpy()
lon = pings["vehicle.position.longitude"].to_numpy()

rows = []
for _, s in st.dropna(subset=["stop_lat"]).iterrows():
    dist = haversine_m(lat, lon, s["stop_lat"], s["stop_lon"])
    i = dist.argmin()
    rows.append({"stop_sequence": s["stop_sequence"], "stop_id": s["stop_id"],
                 "scheduled": s["scheduled"],
                 "actual": pings.loc[i, "vehicle.timestamp"],
                 "nearest_ping_m": round(dist[i])})

res = pd.DataFrame(rows)
res["delay_s"] = (res["actual"] - res["scheduled"]).dt.total_seconds()
res

,stop_sequence,stop_id,scheduled,actual,nearest_ping_m,delay_s
0,1,400751,2026-10-15 20:33:00+00:00,2026-10-15 20:47:22+00:00,163,862.0
1,2,403036,2026-10-15 20:34:37+00:00,2026-10-15 20:47:22+00:00,35,765.0
2,3,403037,2026-10-15 20:36:42+00:00,2026-10-15 20:50:32+00:00,54,830.0
3,4,403038,2026-10-15 20:38:00+00:00,2026-10-15 20:52:19+00:00,63,859.0
4,5,404826,2026-10-15 20:39:28+00:00,2026-10-15 20:53:39+00:00,34,851.0
5,6,405565,2026-10-15 20:40:50+00:00,2026-10-15 20:55:27+00:00,58,877.0
6,7,403626,2026-10-15 20:42:13+00:00,2026-10-15 20:56:47+00:00,47,874.0
7,8,402205,2026-10-15 20:43:13+00:00,2026-10-15 20:57:41+00:00,36,868.0
8,9,403992,2026-10-15 20:45:21+00:00,2026-10-15 20:59:29+00:00,16,848.0
9,10,403984,2026-10-15 20:47:14+00:00,2026-10-15 21:01:44+00:00,20,870.0
